In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque

In [2]:

# name mapping
def map_names(did):
    """ Re-map cells to use their 'name' given their 'did'. Only applies to a
        few select cells where the tracker uses their 'name' instead of 'did'.
    """
    if   did == "P4a": return "Z3"
    elif did == "P4p": return "Z2"
    elif did == "P0a": return "AB"
    else: return did

untracked_nodes = ["AB", "P0", "P1"]

first_internal_layer = ["ABa", "ABp", "EMS", "P2"]

In [3]:
import json

def load_json(file_path):
    """
    Load a JSON file and return its content as a Python object.
    
    :param file_path: Path to the JSON file.
    :return: Parsed JSON content as a Python object.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
    
lineage_data = load_json('./data/cell_lineage.json')


In [4]:
ts_df = pd.read_csv("./data/tracks.txt", sep="\t")
ts_df = ts_df.loc[ts_df["t"] <= 242]
cell_names = ts_df["name"].unique()
valid_cell_names = []
for name in cell_names:
    time_points = ts_df.loc[ts_df['name'] == name]["t"].values
    if len(time_points) == 1 and time_points[0] == 242:
        continue
    valid_cell_names.append(name)

lineage_exp = pd.read_csv("./data/protein/aggregated/lineage_raw_expression.csv", index_col=0)



In [11]:
first_ones = ts_df[ts_df["parent_id"] == -1]
first_ones.mean(axis=0, numeric_only=True)

t              0.00
z            138.00
y            114.75
x            203.25
cell_id        1.50
parent_id     -1.00
track_id       1.50
radius        14.50
div_state      0.00
dtype: float64

In [ ]:
first_ones[["x", "y", "z"]]

,t,z,y,x,cell_id,parent_id,track_id,radius,name,div_state
0,0,144.0,126.0,109.0,3,-1,3,18.0,ABa,0
1,0,149.0,72.0,184.0,1,-1,1,16.5,ABp,0
2,0,132.0,123.0,292.0,2,-1,2,10.0,P2,0
3,0,127.0,138.0,228.0,0,-1,0,13.5,EMS,0


In [7]:
def find_equidistant_point(p1, p2, p3, p4):
    """
    Finds the point equidistant from four given 3D points.
    This is the circumcenter of the tetrahedron defined by the four points.

    :param p1, p2, p3, p4: NumPy arrays or tuples representing the four points in 3D space.
    :return: A NumPy array representing the equidistant point (circumcenter).
             Returns None if the points are coplanar (no unique solution).
    """
    p1, p2, p3, p4 = np.array(p1), np.array(p2), np.array(p3), np.array(p4)

    # Create the matrix A for the system of linear equations
    A = np.array([
        [2 * (p2[0] - p1[0]), 2 * (p2[1] - p1[1]), 2 * (p2[2] - p1[2])],
        [2 * (p3[0] - p1[0]), 2 * (p3[1] - p1[1]), 2 * (p3[2] - p1[2])],
        [2 * (p4[0] - p1[0]), 2 * (p4[1] - p1[1]), 2 * (p4[2] - p1[2])]
    ])

    # Create the vector B
    B = np.array([
        np.sum(p2**2) - np.sum(p1**2),
        np.sum(p3**2) - np.sum(p1**2),
        np.sum(p4**2) - np.sum(p1**2)
    ])

    try:
        # Solve the linear system Ax = B for x
        circumcenter = np.linalg.solve(A, B)
        return circumcenter
    except np.linalg.LinAlgError:
        # This occurs if the matrix A is singular, which means the points are
        # coplanar (or collinear), and there is no unique circumcenter.
        return None

# Example usage with 4 points from the first_internal_layer at t=0
# Get the coordinates for the first four cells at time t=0
p1_coords = ts_df[(ts_df['name'] == 'ABa') & (ts_df['t'] == 0)][['x', 'y', 'z']].iloc[0]
p2_coords = ts_df[(ts_df['name'] == 'ABp') & (ts_df['t'] == 0)][['x', 'y', 'z']].iloc[0]
p3_coords = ts_df[(ts_df['name'] == 'EMS') & (ts_df['t'] == 0)][['x', 'y', 'z']].iloc[0]
p4_coords = ts_df[(ts_df['name'] == 'P2') & (ts_df['t'] == 0)][['x', 'y', 'z']].iloc[0]

# Find the point equidistant to these four points
equidistant_point = find_equidistant_point(p1_coords, p2_coords, p3_coords, p4_coords)

print("Coordinates of the four points:")
print(f"P1 (ABa): {p1_coords.values}")
print(f"P2 (ABp): {p2_coords.values}")
print(f"P3 (EMS): {p3_coords.values}")
print(f"P4 (P2):  {p4_coords.values}")
print("\nCalculated equidistant point (circumcenter):")
print(equidistant_point)

# Optional: Verify the distances
if equidistant_point is not None:
    dist1 = np.linalg.norm(equidistant_point - p1_coords)
    dist2 = np.linalg.norm(equidistant_point - p2_coords)
    dist3 = np.linalg.norm(equidistant_point - p3_coords)
    dist4 = np.linalg.norm(equidistant_point - p4_coords)
    print("\nDistances from the circumcenter to each point:")
    print(f"Distance to P1: {dist1:.4f}")
    print(f"Distance to P2: {dist2:.4f}")
    print(f"Distance to P3: {dist3:.4f}")
    print(f"Distance to P4: {dist4:.4f}")

Coordinates of the four points:
P1 (ABa): [109. 126. 144.]
P2 (ABp): [184.  72. 149.]
P3 (EMS): [228. 138. 127.]
P4 (P2):  [292. 123. 132.]

Calculated equidistant point (circumcenter):
[246.95945946 299.2962578  802.80769231]

Distances from the circumcenter to each point:
Distance to P1: 695.0482
Distance to P2: 695.0482
Distance to P3: 695.0482
Distance to P4: 695.0482
